# Aula 7 — BERT e HuggingFace (Keras/TensorFlow)

Disciplina **CCM-109 Deep Learning** — Prof. Ronaldo Prati (UFABC)

**Tópicos:**
1. MLM Demo — preenchendo lacunas com BERT
2. Embeddings de texto — semântica no espaço vetorial
3. Embeddings não-textuais — imagens e tabelas
4. Visualização de atenção BERT
5. Fine-tuning para classificação de sentimento (TF/Keras)

In [ ]:
# Instalar dependências
!pip install transformers[tf] datasets sentence-transformers --quiet

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {bool(tf.config.list_physical_devices("GPU"))}')

---
## Parte 1 — MLM Demo: BERT prevê tokens mascarados

O `pipeline` do HuggingFace abstrai completamente o backend (PyTorch ou TF).
Vamos explorar o BERT pré-treinado via `fill-mask`.

In [ ]:
from transformers import pipeline, TFBertForMaskedLM, AutoTokenizer as _AutoTok

# Instanciar o modelo TF explicitamente (parametro framework foi removido na API recente)
_mlm_tokenizer = _AutoTok.from_pretrained('bert-base-uncased')
_mlm_model     = TFBertForMaskedLM.from_pretrained('bert-base-uncased')
fill_mask = pipeline('fill-mask', model=_mlm_model, tokenizer=_mlm_tokenizer)

In [ ]:
def show_predictions(text, top_k=5):
    results = fill_mask(text, top_k=top_k)
    print(f'Texto: "{text}"\n')
    print(f'{"Token":20s}  {"Score":>8s}  Frase')
    print('-' * 60)
    for r in results:
        print(f"{r['token_str']:20s}  {r['score']:8.4f}  {r['sequence']}")
    print()

# Contexto desambigua o sentido de "bank"
show_predictions("She went to the bank to [MASK] money.")
show_predictions("She sat on the river [MASK] and watched the sunset.")

In [ ]:
# Visualização interativa: distribuição para múltiplas frases
examples = [
    "The [MASK] is the largest planet in the solar system.",
    "Python is a popular programming [MASK].",
    "Deep learning models are trained using [MASK] descent.",
]

fig, axes = plt.subplots(1, len(examples), figsize=(15, 4))

for ax, text in zip(axes, examples):
    results = fill_mask(text, top_k=8)
    tokens  = [r['token_str'] for r in results]
    scores  = [r['score'] for r in results]
    colors  = plt.cm.Blues(np.linspace(0.4, 0.9, len(tokens)))
    
    ax.barh(tokens[::-1], scores[::-1], color=colors[::-1])
    ax.set_xlabel('P(token)')
    masked_part = text[text.find('['):text.find(']')+1]
    ax.set_title(f'...{text[max(0,text.find("[")-15):text.find("]")+1]}...', fontsize=9)

plt.suptitle('BERT fill-mask: distribuição sobre o vocabulário', fontsize=11)
plt.tight_layout()
plt.show()

---
## Parte 2 — Embeddings de texto com TFBertModel

Extraímos embeddings diretamente do `TFBertModel` com TensorFlow.

In [ ]:
from transformers import AutoTokenizer, TFAutoModel

model_name = 'bert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
bert_tf    = TFAutoModel.from_pretrained(model_name)
print('TFBertModel carregado.')

In [ ]:
def get_bert_embedding_tf(texts, pooling='mean'):
    """Extrai embeddings usando TFBertModel."""
    inputs = tokenizer(texts, return_tensors='tf', padding=True,
                       truncation=True, max_length=128)
    outputs = bert_tf(**inputs, training=False)
    last_hidden = outputs.last_hidden_state  # (B, T, 768)
    
    if pooling == 'cls':
        return last_hidden[:, 0, :].numpy()
    else:  # mean pooling
        mask = tf.cast(inputs['attention_mask'][:, :, tf.newaxis], tf.float32)
        return (tf.reduce_sum(last_hidden * mask, axis=1) /
                tf.reduce_sum(mask, axis=1)).numpy()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

sentences = [
    'Python is a programming language used for machine learning.',
    'Deep learning requires large amounts of training data.',
    'Neural networks learn hierarchical representations.',
    'The soccer team won the championship last night.',
    'Basketball players need excellent coordination and speed.',
    'The marathon runner finished the race in record time.',
    'The pasta was cooked with olive oil and fresh tomatoes.',
    'Baking bread requires flour, yeast, water, and salt.',
    'The chef prepared a delicious chocolate cake for dessert.',
]
categories = ['Tech']*3 + ['Sport']*3 + ['Food']*3
color_map = {'Tech': '#3b82f6', 'Sport': '#ef4444', 'Food': '#10b981'}

embs = get_bert_embedding_tf(sentences, pooling='mean')
print(f'Embeddings shape: {embs.shape}')

In [ ]:
pca    = PCA(n_components=2)
embs2d = pca.fit_transform(embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
ax = axes[0]
for i, (x, y) in enumerate(embs2d):
    color = color_map[categories[i]]
    ax.scatter(x, y, color=color, s=120, zorder=3)
    ax.annotate(sentences[i][:32] + '…', (x, y),
                textcoords='offset points', xytext=(6, 3), fontsize=7)
for cat, col in color_map.items():
    ax.scatter([], [], color=col, label=cat, s=80)
ax.legend()
ax.set_title('PCA 2D — TFBertModel (mean pooling)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.grid(alpha=0.3)

# Cosine similarity
ax2 = axes[1]
embs_n = normalize(embs)
sim    = embs_n @ embs_n.T
short  = [s[:28]+'…' for s in sentences]
im = ax2.imshow(sim, cmap='RdYlGn', vmin=0.5, vmax=1.0)
ax2.set_xticks(range(9)); ax2.set_xticklabels(short, rotation=45, ha='right', fontsize=7)
ax2.set_yticks(range(9)); ax2.set_yticklabels(short, fontsize=7)
for i in range(9):
    for j in range(9):
        ax2.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax2)
ax2.set_title('Similaridade coseno')

plt.tight_layout()
plt.show()

---
## Parte 3 — Embeddings não-textuais

### 3.1 Sentence-BERT vs BERT-mean para recuperação semântica

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2')

query = "How to train a machine learning model?"
candidates = [
    "Steps to fit a neural network on data.",
    "The weather was sunny and warm today.",
    "Training procedures for deep learning systems.",
    "My dog loves playing fetch in the park.",
    "Gradient descent optimizes model parameters.",
]

# BERT mean pooling
q_bert = get_bert_embedding_tf([query], pooling='mean')
c_bert = get_bert_embedding_tf(candidates, pooling='mean')
sim_bert = (normalize(q_bert) @ normalize(c_bert).T)[0]

# Sentence-BERT
q_sbert = sbert.encode([query])
c_sbert = sbert.encode(candidates)
sim_sbert = (normalize(q_sbert) @ normalize(c_sbert).T)[0]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(candidates))
w = 0.35
ax.bar(x - w/2, sim_bert,  w, label='BERT (TF mean pool)', color='#6366f1', alpha=0.8)
ax.bar(x + w/2, sim_sbert, w, label='Sentence-BERT',       color='#10b981', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([c[:32]+'…' for c in candidates], rotation=18, ha='right', fontsize=8)
ax.set_ylabel('Similaridade coseno')
ax.set_title(f'Query: "{query}"')
ax.legend(); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Embeddings tabulares com Keras

Variáveis categóricas → `tf.keras.layers.Embedding` → vetores densos treináveis

In [ ]:
import keras

# Dataset sintético: prever risco de churn
np.random.seed(42)
N = 500

plano_idx     = np.random.randint(0, 3, N)   # basic=0, standard=1, premium=2
regiao_idx    = np.random.randint(0, 5, N)   # 5 regiões
pagamento_idx = np.random.randint(0, 3, N)   # 3 métodos

# Feature numérica: meses de contrato (normalizada)
meses = np.random.exponential(12, N).clip(1, 60)
valor = np.random.normal(150, 50, N).clip(50, 300)
numericas = np.stack([meses / 60, valor / 300], axis=1).astype(np.float32)

# Label sintética: plano básico + pagamento boleto = maior churn
churn_prob = (0.3 * (plano_idx == 0) + 0.2 * (pagamento_idx == 2) +
              0.1 * np.random.rand(N))
labels = (churn_prob > 0.4).astype(np.float32)
print(f'Taxa de churn: {labels.mean():.1%}')

In [ ]:
EMB_DIM = 8

# Entradas categóricas
inp_plano     = keras.Input(shape=(1,), dtype='int32', name='plano')
inp_regiao    = keras.Input(shape=(1,), dtype='int32', name='regiao')
inp_pagamento = keras.Input(shape=(1,), dtype='int32', name='pagamento')
inp_numericas = keras.Input(shape=(2,), dtype='float32', name='numericas')

# Embeddings
e_plano     = keras.layers.Embedding(3, EMB_DIM, name='emb_plano')(inp_plano)
e_regiao    = keras.layers.Embedding(5, EMB_DIM, name='emb_regiao')(inp_regiao)
e_pagamento = keras.layers.Embedding(3, EMB_DIM, name='emb_pagamento')(inp_pagamento)

# Flatten embeddings e concatenar com numéricas
flat_plano     = keras.layers.Flatten()(e_plano)
flat_regiao    = keras.layers.Flatten()(e_regiao)
flat_pagamento = keras.layers.Flatten()(e_pagamento)

x = keras.layers.Concatenate()([flat_plano, flat_regiao, flat_pagamento, inp_numericas])
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(32, activation='relu')(x)
output = keras.layers.Dense(1, activation='sigmoid')(x)

tab_model = keras.Model(
    inputs=[inp_plano, inp_regiao, inp_pagamento, inp_numericas],
    outputs=output
)

tab_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
tab_model.summary()

In [ ]:
history_tab = tab_model.fit(
    {'plano': plano_idx, 'regiao': regiao_idx,
     'pagamento': pagamento_idx, 'numericas': numericas},
    labels,
    epochs=20, batch_size=32, validation_split=0.2, verbose=0
)

final_acc = history_tab.history['val_accuracy'][-1]
print(f'Val accuracy final: {final_acc:.4f}')

# Visualizar embeddings aprendidos
emb_plano_aprendido = tab_model.get_layer('emb_plano').get_weights()[0]  # (3, 8)

fig, ax = plt.subplots(figsize=(6, 4))
plano_2d = PCA(n_components=2).fit_transform(emb_plano_aprendido)
colors   = ['#ef4444', '#f97316', '#10b981']
nomes    = ['basic', 'standard', 'premium']
for i, (nome, cor) in enumerate(zip(nomes, colors)):
    ax.scatter(*plano_2d[i], color=cor, s=300, zorder=3)
    ax.annotate(nome, plano_2d[i], textcoords='offset points',
                xytext=(8, 4), fontsize=12, fontweight='bold')
ax.set_title('Embeddings de "plano" aprendidos (PCA 2D)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Parte 4 — Visualização de atenção BERT (TF)

Com `TFBertModel(output_attentions=True)`, extraímos os pesos de atenção de cada camada e cabeça.

In [ ]:
from transformers import TFBertModel

bert_attn_tf = TFBertModel.from_pretrained('bert-base-uncased',
                                            output_attentions=True)
print('TFBertModel com atenções carregado.')

In [ ]:
def get_attention_tf(text, layer=11):
    inputs = tokenizer(text, return_tensors='tf')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].numpy())
    outputs = bert_attn_tf(**inputs, training=False)
    # attentions: lista de 12 tensores (1, num_heads, T, T)
    attn = outputs.attentions[layer][0].numpy()  # (12, T, T)
    return tokens, attn

sentence = "The bank can guarantee deposits will eventually cover future tuition costs."
tokens, attn_last = get_attention_tf(sentence, layer=11)
print(f'Tokens: {tokens}')
print(f'Atenção shape: {attn_last.shape}')

In [ ]:
# Comparação entre camadas: atenção média das 12 cabeças
_, attn_c1  = get_attention_tf(sentence, layer=0)
_, attn_c6  = get_attention_tf(sentence, layer=5)
_, attn_c12 = get_attention_tf(sentence, layer=11)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (attn, title) in zip(axes, [
    (attn_c1.mean(0),  'Camada 1'),
    (attn_c6.mean(0),  'Camada 6'),
    (attn_c12.mean(0), 'Camada 12'),
]):
    im = ax.imshow(attn, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'{title} — média das 12 cabeças')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)

plt.suptitle(f'"{sentence[:60]}…"', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Detalhe: 6 cabeças da última camada
n_heads = 6
T = len(tokens)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for h in range(n_heads):
    ax = axes[h]
    im = ax.imshow(attn_c12[h], cmap='Blues', aspect='auto')
    ax.set_xticks(range(T))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(T))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'Cabeça {h+1}')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Pesos de atenção — Camada 12 (6 cabeças)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Parte 5 — Fine-tuning com Keras: análise de sentimento SST-2

Usaremos a API `TFBertForSequenceClassification` (modelo BERT + cabeça de classificação integrada) com o `Trainer` do HuggingFace ou com Keras puro.

In [ ]:
from transformers import TFAutoModelForSequenceClassification, DataCollatorWithPadding
from datasets import load_dataset

dataset = load_dataset('glue', 'sst2')

# Tokenizar
def tokenize(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=64)

tokenized = dataset.map(tokenize, batched=True)
print(tokenized)

In [ ]:
# Subsample para demonstração rápida
train_sub = tokenized['train'].shuffle(seed=42).select(range(2000))
val_sub   = tokenized['validation']

# Converter para tf.data.Dataset
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='tf')

BATCH = 32
train_tf = train_sub.to_tf_dataset(
    columns=['input_ids', 'attention_mask', 'token_type_ids'],
    label_cols=['label'],
    shuffle=True,
    batch_size=BATCH,
    collate_fn=collator
)
val_tf = val_sub.to_tf_dataset(
    columns=['input_ids', 'attention_mask', 'token_type_ids'],
    label_cols=['label'],
    shuffle=False,
    batch_size=BATCH,
    collate_fn=collator
)
print('tf.data.Datasets criados.')

In [ ]:
# Modelo: BERT + cabeça de classificação integrada
bert_clf = TFAutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

EPOCHS  = 3
LR      = 2e-5
N_STEPS = len(train_sub) // BATCH

# Warmup linear + decay
from transformers import create_optimizer
optimizer_tf, lr_schedule = create_optimizer(
    init_lr=LR,
    num_train_steps=N_STEPS * EPOCHS,
    num_warmup_steps=N_STEPS * EPOCHS // 10,
    weight_decay_rate=0.01
)

bert_clf.compile(
    optimizer=optimizer_tf,
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print(f'Parâmetros totais: {bert_clf.count_params():,}')

In [ ]:
history_bert = bert_clf.fit(
    train_tf,
    validation_data=val_tf,
    epochs=EPOCHS
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, history_bert.history['loss'],     'o-', label='Train', color='#6366f1')
ax1.plot(epochs, history_bert.history['val_loss'], 's-', label='Val',   color='#ef4444')
ax1.set_title('Loss'); ax1.set_xlabel('Época'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history_bert.history['accuracy'],     'o-', label='Train', color='#6366f1')
ax2.plot(epochs, history_bert.history['val_accuracy'], 's-', label='Val',   color='#ef4444')
ax2.set_title('Acurácia'); ax2.set_xlabel('Época'); ax2.set_ylabel('Acurácia')
ax2.legend(); ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)

plt.suptitle('Fine-tuning BERT (TF/Keras) — SST-2', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Inferência
def predict_sentiment_tf(texts):
    inputs = tokenizer(texts, return_tensors='tf', padding=True,
                       truncation=True, max_length=64)
    logits = bert_clf(**inputs, training=False).logits
    probs  = tf.nn.softmax(logits, axis=-1).numpy()
    labels = ['negativo', 'positivo']
    for text, prob in zip(texts, probs):
        pred = labels[prob.argmax()]
        print(f'[{pred} {prob.max():.2%}] {text[:70]}')

predict_sentiment_tf([
    "This movie was absolutely fantastic and I loved every minute of it!",
    "The film was a complete waste of time, terribly boring.",
    "It was okay, nothing special but not terrible either.",
    "One of the best performances I have ever seen on screen.",
])

---
## Resumo

| Tópico | API Keras/TF usada |
|---|---|
| MLM demo | `` |
| Embeddings texto | `TFAutoModel` + mean pooling |
| Embeddings tabulares | `keras.layers.Embedding` treinável |
| Visualização atenção | `TFBertModel(output_attentions=True)` |
| Fine-tuning SST-2 | `TFAutoModelForSequenceClassification` + `create_optimizer` |

**Comparação PyTorch vs Keras:**
- PyTorch: mais controle (loop explícito, `GradScaler`, scheduler passo a passo)
- Keras: mais conciso (`model.fit`, `to_tf_dataset`, `create_optimizer` com warmup integrado)